# VisionTrack — Complete ML / Computer Vision Notebook

**This notebook is the single source of truth for the Python ML/CV code.**

There are intentionally **no standalone `.py` files** for the ML pipeline or the
API layer. Everything from model loading to the FastAPI + MongoDB service that
the React dashboard talks to lives in this notebook.

Pipeline:

**Image/Webcam → preprocessing → SSD MobileNet V2 → confidence filtering → pixel boxes → ByteTrack → object analytics → line crossing → annotated output → MongoDB session persistence**

## How this notebook is used in the project

| Mode (`VISIONTRACK_MODE`) | What happens | Where it runs |
|---|---|---|
| `notebook` (default) | Cells run normally for experimentation; the demo/API cells stay idle. | Local Jupyter |
| `webcam` | Section 13 opens your local webcam in an OpenCV window. | Local Jupyter (needs a camera) |
| `api` | Section 17 starts the FastAPI + MongoDB backend and serves the React app. | Docker container (headless, via `jupyter nbconvert --execute`) |

The rest of the project (React frontend, MongoDB, Docker, system design docs)
lives alongside this notebook in the repository — see `README.md` and
`docs/SYSTEM_DESIGN.md`.

## 1. Install dependencies

Run this once in the Jupyter environment. (The Docker image installs the same
list from `notebooks/requirements.txt`, which is generated from this cell plus
the headless-execution tools `jupyter`, `nbconvert`, and `ipykernel`.)

In [ ]:
%pip install tensorflow tensorflow-hub opencv-python numpy supervision trackers matplotlib pymongo fastapi uvicorn python-multipart


## 2. Imports and configuration

We keep the model, tracking, and **service** configuration in one place so
experiments are reproducible and so the same notebook can run interactively
in Jupyter or headlessly as the backend container. Everything is overridable
with environment variables so Docker/Compose can configure it without editing
the notebook.

In [ ]:
import os
import time
from collections import Counter
from datetime import datetime, timezone

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_hub as hub
import supervision as sv
import uvicorn

from trackers import ByteTrackTracker

# --- ML / tracking configuration ---
MODEL_URL = "https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2"
CONFIDENCE_THRESHOLD = float(os.environ.get("VISIONTRACK_CONFIDENCE_THRESHOLD", 0.5))
TRACKER_FRAME_RATE = float(os.environ.get("VISIONTRACK_TRACKER_FRAME_RATE", 3.0))
LINE_Y = int(os.environ.get("VISIONTRACK_LINE_Y", 300))
CAMERA_INDEX = int(os.environ.get("VISIONTRACK_CAMERA_INDEX", 0))

# --- Service configuration (backend + MongoDB) ---
# "notebook" = normal interactive use, "webcam" = local camera demo,
# "api" = start the FastAPI + MongoDB service (used by the Docker image).
RUN_MODE = os.environ.get("VISIONTRACK_MODE", "notebook")
API_PORT = int(os.environ.get("API_PORT", 8000))
MONGO_URI = os.environ.get("MONGO_URI", "mongodb://localhost:27017")
MONGO_DB = os.environ.get("MONGO_DB", "visiontrack")
SNAPSHOT_INTERVAL_SECONDS = float(os.environ.get("SNAPSHOT_INTERVAL_SECONDS", 2.0))
CORS_ORIGINS = [
    origin.strip()
    for origin in os.environ.get(
        "CORS_ORIGINS", "http://localhost:5173,http://localhost"
    ).split(",")
    if origin.strip()
]

print("TensorFlow:", tf.__version__)
print("Run mode:", RUN_MODE)


## 3. COCO class labels

SSD MobileNet V2 is pretrained on COCO. These labels map model class IDs to
human-readable names. This is the **single source of truth** for label names
in the whole project — the FastAPI layer below resolves names from this dict
before sending analytics to the React frontend, so the frontend never has to
keep its own copy in sync.

In [ ]:
COCO_LABELS = {
    1: "person", 2: "bicycle", 3: "car", 4: "motorcycle",
    5: "airplane", 6: "bus", 7: "train", 8: "truck", 9: "boat",
    17: "cat", 18: "dog", 19: "horse", 44: "bottle", 47: "cup",
    62: "chair", 63: "couch", 64: "potted plant", 67: "dining table",
    73: "laptop", 77: "cell phone",
}


## 4. Load the pretrained detector

The project does **not** train SSD MobileNet V2 from scratch. It integrates a
pretrained COCO detector and focuses on real-time inference, tracking,
analytics, APIs, and system engineering.

In [ ]:
print("Loading SSD MobileNet V2...")
detector_model = hub.load(MODEL_URL)
print("Model loaded successfully.")


## 5. Detection

TensorFlow returns normalized boxes in:

`[y_min, x_min, y_max, x_max]`

We filter detections using a confidence threshold.

In [ ]:
def detect(frame, confidence_threshold=CONFIDENCE_THRESHOLD):
    rgb_frame = tf.convert_to_tensor(frame[..., ::-1])
    input_tensor = tf.expand_dims(rgb_frame, axis=0)

    detections = detector_model(input_tensor)

    scores = detections["detection_scores"][0]
    boxes = detections["detection_boxes"][0]
    classes = detections["detection_classes"][0]

    valid = scores >= confidence_threshold

    return {
        "boxes": tf.boolean_mask(boxes, valid).numpy(),
        "scores": tf.boolean_mask(scores, valid).numpy(),
        "classes": tf.boolean_mask(classes, valid).numpy(),
    }


## 6. Convert normalized boxes to pixel coordinates

The tracker and OpenCV drawing functions need pixel coordinates in:

`[x1, y1, x2, y2]`

In [ ]:
def to_pixel_boxes(normalized_boxes, frame_width, frame_height):
    pixel_boxes = []

    for y_min, x_min, y_max, x_max in normalized_boxes:
        pixel_boxes.append([
            x_min * frame_width,
            y_min * frame_height,
            x_max * frame_width,
            y_max * frame_height,
        ])

    if pixel_boxes:
        return np.asarray(pixel_boxes, dtype=np.float32)

    return np.empty((0, 4), dtype=np.float32)


## 7. ByteTrack multi-object tracking

The detector answers **"what objects are in this frame?"**

ByteTrack answers **"which detection belongs to the same object across frames?"**

A tracker ID lets us count objects and detect movement across a line.

In [ ]:
class ObjectTracker:
    def __init__(self, frame_rate=TRACKER_FRAME_RATE):
        self.tracker = ByteTrackTracker(frame_rate=frame_rate)

    def update(self, boxes, scores, classes):
        detections = sv.Detections(
            xyxy=boxes,
            confidence=scores,
            class_id=classes.astype(np.int32),
        )
        return self.tracker.update(detections)


## 8. Object analytics

We count active tracked objects and break them down by class.

In [ ]:
class ObjectCounter:
    def count(self, tracked_detections):
        if tracked_detections.tracker_id is None:
            return 0

        valid_ids = tracked_detections.tracker_id[
            tracked_detections.tracker_id >= 0
        ]
        return len(valid_ids)

    def count_by_class(self, tracked_detections):
        if (
            tracked_detections.tracker_id is None
            or tracked_detections.class_id is None
        ):
            return {}

        valid_mask = tracked_detections.tracker_id >= 0
        valid_classes = tracked_detections.class_id[valid_mask]

        return dict(
            Counter(int(class_id) for class_id in valid_classes)
        )


## 9. Line-crossing analytics

An object crossing downward through the configured horizontal line is counted
as **entered**.

An object crossing upward is counted as **exited**.

In [ ]:
class LineCounter:
    def __init__(self, line_y):
        self.line_y = line_y
        self.previous_positions = {}
        self.entered = 0
        self.exited = 0

    def update(self, tracked_detections):
        if tracked_detections.tracker_id is None:
            return

        for box, tracker_id in zip(
            tracked_detections.xyxy,
            tracked_detections.tracker_id,
        ):
            tracker_id = int(tracker_id)

            if tracker_id < 0:
                continue

            _, y_min, _, y_max = box
            center_y = int((y_min + y_max) / 2)

            previous_y = self.previous_positions.get(tracker_id)

            if previous_y is not None:
                if previous_y < self.line_y <= center_y:
                    self.entered += 1
                elif previous_y > self.line_y >= center_y:
                    self.exited += 1

            self.previous_positions[tracker_id] = center_y


## 10. Annotation

This function draws bounding boxes, class labels, tracker IDs, the counting
line, and FPS.

In [ ]:
def annotate_frame(frame, tracked, fps, line_y=LINE_Y):
    output = frame.copy()
    frame_height, frame_width = output.shape[:2]

    tracker_ids = tracked.tracker_id
    if tracker_ids is None:
        tracker_ids = np.full(len(tracked.xyxy), -1, dtype=np.int32)

    if tracked.xyxy is not None:
        for box, score, class_id, tracker_id in zip(
            tracked.xyxy,
            tracked.confidence,
            tracked.class_id,
            tracker_ids,
        ):
            x1, y1, x2, y2 = map(int, box)

            x1 = max(0, min(x1, frame_width - 1))
            y1 = max(0, min(y1, frame_height - 1))
            x2 = max(0, min(x2, frame_width - 1))
            y2 = max(0, min(y2, frame_height - 1))

            class_id = int(class_id)
            tracker_id = int(tracker_id)
            score = float(score)

            label = COCO_LABELS.get(
                class_id,
                f"class_{class_id}",
            )

            if tracker_id >= 0:
                text = f"{label} {score:.2f} ID:{tracker_id}"
            else:
                text = f"{label} {score:.2f}"

            cv2.rectangle(
                output,
                (x1, y1),
                (x2, y2),
                (255, 0, 0),
                2,
            )

            cv2.putText(
                output,
                text,
                (x1, max(25, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

    safe_line_y = min(line_y, frame_height - 1)

    cv2.line(
        output,
        (0, safe_line_y),
        (frame_width, safe_line_y),
        (0, 255, 255),
        2,
    )

    cv2.putText(
        output,
        f"FPS: {fps:.1f}",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2,
        cv2.LINE_AA,
    )

    return output


## 11. End-to-end frame processing

This is the main ML pipeline: detection → coordinate conversion → tracking →
analytics → annotation. It returns the raw per-frame track/class IDs as well
as the aggregated counts, so the session layer below (Section 16) can work
out how many **distinct** objects were seen over an entire session, not just
what's visible in the current frame.

In [ ]:
tracker = ObjectTracker()
counter = ObjectCounter()
line_counter = LineCounter(LINE_Y)

def process_frame(frame):
    start = time.perf_counter()

    frame_height, frame_width = frame.shape[:2]

    detections = detect(frame)

    pixel_boxes = to_pixel_boxes(
        detections["boxes"],
        frame_width,
        frame_height,
    )

    tracked = tracker.update(
        pixel_boxes,
        detections["scores"],
        detections["classes"],
    )

    line_counter.update(tracked)

    active_objects = counter.count(tracked)
    class_counts = counter.count_by_class(tracked)

    elapsed = time.perf_counter() - start
    fps = 1.0 / elapsed if elapsed > 0 else 0.0

    annotated = annotate_frame(
        frame,
        tracked,
        fps,
        LINE_Y,
    )

    track_ids = (
        tracked.tracker_id.tolist()
        if tracked.tracker_id is not None
        else []
    )
    active_class_ids = (
        tracked.class_id.tolist()
        if tracked.class_id is not None
        else []
    )

    return {
        "frame": annotated,
        "active_objects": active_objects,
        "fps": round(fps, 2),
        "entered": line_counter.entered,
        "exited": line_counter.exited,
        "class_counts": class_counts,
        "track_ids": track_ids,
        "active_class_ids": active_class_ids,
    }


## 12. Test on an image

Set `IMAGE_PATH` to any local image. This section is useful for explaining
the detector before moving to the webcam.

In [ ]:
IMAGE_PATH = "sample.jpg"

image = cv2.imread(IMAGE_PATH)

if image is None:
    print("Put an image at IMAGE_PATH before running this cell.")
else:
    result = process_frame(image)

    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(result["frame"], cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(
        f"Objects: {result['active_objects']} | "
        f"FPS: {result['fps']}"
    )
    plt.show()

    print(result["class_counts"])


## 13. Live webcam demo

Run this cell locally with `VISIONTRACK_MODE=webcam`. Press **Q** in the
OpenCV window to stop.

This is the ML notebook's standalone webcam mode; the React dashboard is a
separate frontend that talks to Section 17's API instead. The call is guarded
by `RUN_MODE` so that headlessly executing this notebook in Docker (where
there is no camera and no display) doesn't crash.

In [ ]:
def run_webcam(camera_index=CAMERA_INDEX):
    cap = cv2.VideoCapture(camera_index)

    if not cap.isOpened():
        raise RuntimeError(
            "Could not open webcam. Check browser/OS camera permissions "
            "and make sure another application is not using the camera."
        )

    try:
        while True:
            ok, frame = cap.read()

            if not ok:
                print("Could not read a webcam frame.")
                break

            result = process_frame(frame)

            cv2.imshow(
                "VisionTrack - ML Notebook",
                result["frame"],
            )

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    finally:
        cap.release()
        cv2.destroyAllWindows()

if RUN_MODE == "webcam":
    run_webcam()
else:
    print(
        f"Skipping the live webcam demo because VISIONTRACK_MODE=\'{RUN_MODE}\'. "
        "Set the VISIONTRACK_MODE environment variable to \'webcam\' and re-run "
        "this cell (from a local Jupyter kernel with camera access) to use it."
    )


## 14. Session management & MongoDB persistence

The React dashboard drives one live camera session at a time: **start
camera → stream frames → stop camera**. We keep per-frame state in memory
(fast, no database round trip per frame) and only touch MongoDB for:

1. **Throttled snapshots** — a lightweight time-series point every
   `SNAPSHOT_INTERVAL_SECONDS` (default 2s), so a session's history can be
   charted without writing to Mongo on every single frame (which, even at a
   modest 5–10 FPS, would be an unnecessary write for every frame).
2. **One summary document per session**, written once when the session
   stops.

`SessionManager` also tracks the **set of distinct tracker IDs seen per
class** over the whole session (via the `track_ids` / `active_class_ids`
returned by `process_frame`), which is a far more useful analytic for a
finished session than "whatever was on-screen in the final frame".

In [ ]:
import uuid
from threading import Lock

from pymongo import MongoClient, DESCENDING

mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
db = mongo_client[MONGO_DB]
sessions_collection = db["sessions"]
snapshots_collection = db["session_snapshots"]


def mongo_is_connected():
    try:
        mongo_client.admin.command("ping")
        return True
    except Exception:
        return False


class SessionManager:
    """Tracks the single live camera session the dashboard can have open."""

    def __init__(self):
        self._lock = Lock()
        self._reset()

    def _reset(self):
        self.session_id = None
        self.started_at = None
        self.frame_count = 0
        self.fps_sum = 0.0
        self.peak_active_objects = 0
        self.last_entered = 0
        self.last_exited = 0
        self.seen_track_ids_by_class = {}
        self.last_snapshot_at = 0.0

    def ensure_started(self):
        with self._lock:
            if self.session_id is None:
                self._reset()
                self.session_id = str(uuid.uuid4())
                self.started_at = datetime.now(timezone.utc)
            return self.session_id

    def record_frame(self, result):
        """Update in-memory aggregates; return a snapshot dict if one is due."""
        with self._lock:
            if self.session_id is None:
                return None

            self.frame_count += 1
            self.fps_sum += result["fps"]
            self.peak_active_objects = max(
                self.peak_active_objects, result["active_objects"]
            )
            self.last_entered = result["entered"]
            self.last_exited = result["exited"]

            for track_id, class_id in zip(
                result.get("track_ids", []), result.get("active_class_ids", [])
            ):
                bucket = self.seen_track_ids_by_class.setdefault(int(class_id), set())
                bucket.add(int(track_id))

            now = time.monotonic()
            due = (now - self.last_snapshot_at) >= SNAPSHOT_INTERVAL_SECONDS

            if not due:
                return None

            self.last_snapshot_at = now
            return {
                "session_id": self.session_id,
                "timestamp": datetime.now(timezone.utc),
                "active_objects": result["active_objects"],
                "fps": result["fps"],
                "entered": result["entered"],
                "exited": result["exited"],
            }

    def finish(self):
        """Finalize the active session and return its summary (or None)."""
        with self._lock:
            if self.session_id is None:
                return None

            ended_at = datetime.now(timezone.utc)
            duration = (ended_at - self.started_at).total_seconds()
            avg_fps = (
                round(self.fps_sum / self.frame_count, 2)
                if self.frame_count
                else 0.0
            )

            distinct_object_counts = {
                COCO_LABELS.get(class_id, f"class_{class_id}"): len(track_ids)
                for class_id, track_ids in self.seen_track_ids_by_class.items()
            }

            summary = {
                "_id": self.session_id,
                "started_at": self.started_at,
                "ended_at": ended_at,
                "duration_seconds": round(duration, 2),
                "total_frames_processed": self.frame_count,
                "avg_fps": avg_fps,
                "peak_active_objects": self.peak_active_objects,
                "entered": self.last_entered,
                "exited": self.last_exited,
                "distinct_object_counts": distinct_object_counts,
            }

            self._reset()
            return summary


session_manager = SessionManager()


## 15. FastAPI service for the React frontend

The code below is still part of this notebook — **there is no backend `.py`
file**. It exposes the ML pipeline and the session/MongoDB layer above over
HTTP so the React dashboard can drive it.

| Method & path | Purpose |
|---|---|
| `GET /api/health` | Liveness + Mongo connectivity check |
| `POST /api/process-frame` | Run one frame through the ML pipeline, return the annotated JPEG + analytics headers |
| `POST /api/session/stop` | Finalize the active session, persist its summary to MongoDB |
| `GET /api/sessions` | List recent saved sessions (most recent first) |
| `GET /api/sessions/{session_id}` | One session's summary plus its snapshot time series |

In Docker, this section is started automatically (see Section 17). In a
normal Jupyter kernel, running this cell just *defines* the app — nothing
starts listening until Section 17 is run with `VISIONTRACK_MODE=api`.

In [ ]:
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, Response
from starlette.concurrency import run_in_threadpool
import json

api = FastAPI(title="VisionTrack Notebook ML API")

api.add_middleware(
    CORSMiddleware,
    allow_origins=CORS_ORIGINS,
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=[
        "X-Active-Objects",
        "X-FPS",
        "X-Entered",
        "X-Exited",
        "X-Class-Counts",
        "X-Session-Id",
    ],
)


def _serialize_session(doc):
    """Mongo documents carry datetimes and `_id`; make them JSON/JS-friendly."""
    out = dict(doc)
    out["session_id"] = out.pop("_id")
    for key in ("started_at", "ended_at"):
        value = out.get(key)
        if isinstance(value, datetime):
            out[key] = value.isoformat()
    return out


@api.get("/api/health")
def health():
    return {
        "status": "ok",
        "source": "ml-notebook",
        "mongo_connected": mongo_is_connected(),
        "active_session": session_manager.session_id,
    }


@api.post("/api/process-frame")
async def process_api_frame(file: UploadFile = File(...)):
    raw = await file.read()

    if not raw:
        raise HTTPException(status_code=400, detail="Empty upload")

    frame = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)

    if frame is None:
        raise HTTPException(status_code=400, detail="Invalid image")

    session_id = session_manager.ensure_started()

    # Detection/tracking is CPU-bound and synchronous; run it off the event
    # loop so the server can keep accepting requests (health checks, other
    # tabs, etc.) while a frame is being processed.
    result = await run_in_threadpool(process_frame, frame)

    snapshot = session_manager.record_frame(result)
    if snapshot is not None:
        await run_in_threadpool(snapshots_collection.insert_one, snapshot)

    ok, encoded = cv2.imencode(".jpg", result["frame"])

    if not ok:
        raise HTTPException(status_code=500, detail="Could not encode frame")

    named_class_counts = {
        COCO_LABELS.get(class_id, f"class_{class_id}"): count
        for class_id, count in result["class_counts"].items()
    }

    headers = {
        "X-Active-Objects": str(result["active_objects"]),
        "X-FPS": str(result["fps"]),
        "X-Entered": str(result["entered"]),
        "X-Exited": str(result["exited"]),
        "X-Class-Counts": json.dumps(named_class_counts),
        "X-Session-Id": session_id,
    }

    return Response(
        content=encoded.tobytes(),
        media_type="image/jpeg",
        headers=headers,
    )


@api.post("/api/session/stop")
async def stop_session():
    summary = await run_in_threadpool(session_manager.finish)

    if summary is None:
        return JSONResponse({"session_id": None, "message": "No active session"})

    await run_in_threadpool(sessions_collection.insert_one, summary)

    return _serialize_session(summary)


@api.get("/api/sessions")
async def list_sessions(limit: int = 20):
    limit = max(1, min(limit, 100))
    cursor = sessions_collection.find().sort("started_at", DESCENDING).limit(limit)
    docs = await run_in_threadpool(list, cursor)
    return [_serialize_session(doc) for doc in docs]


@api.get("/api/sessions/{session_id}")
async def get_session(session_id: str):
    doc = await run_in_threadpool(sessions_collection.find_one, {"_id": session_id})

    if doc is None:
        raise HTTPException(status_code=404, detail="Session not found")

    snapshots_cursor = snapshots_collection.find(
        {"session_id": session_id}
    ).sort("timestamp", 1)
    snapshots = await run_in_threadpool(list, snapshots_cursor)

    payload = _serialize_session(doc)
    payload["snapshots"] = [
        {
            "timestamp": snap["timestamp"].isoformat(),
            "active_objects": snap["active_objects"],
            "fps": snap["fps"],
            "entered": snap["entered"],
            "exited": snap["exited"],
        }
        for snap in snapshots
    ]
    return payload


@api.delete("/api/sessions/{session_id}")
async def delete_session(session_id: str):
    result = await run_in_threadpool(
        sessions_collection.delete_one, {"_id": session_id}
    )
    await run_in_threadpool(
        snapshots_collection.delete_many, {"session_id": session_id}
    )

    if result.deleted_count == 0:
        raise HTTPException(status_code=404, detail="Session not found")

    return {"deleted": session_id}


## 16. Run the API server

This is what actually turns the notebook into a running backend. It only
starts the server when `VISIONTRACK_MODE=api`, so:

- Running the whole notebook top-to-bottom in normal Jupyter (`notebook`
  mode) is always safe — this cell just prints a message and returns.
- The Docker image sets `VISIONTRACK_MODE=api` and executes the notebook
  headlessly with `jupyter nbconvert --execute`, so this cell is the one
  that actually starts serving requests, and it runs forever (until the
  container stops).

We use `await server.serve()` (Uvicorn's async entrypoint) rather than
`uvicorn.run(...)` because Jupyter kernels already run their own asyncio
event loop — `uvicorn.run()` would try to start a second one and fail. IPython
supports top-level `await` in cells, so this works directly.

In [ ]:
if RUN_MODE == "api":
    print(f"Starting VisionTrack API on 0.0.0.0:{API_PORT}")
    print(f"MongoDB: {MONGO_URI} (db={MONGO_DB}, connected={mongo_is_connected()})")
    print(f"Allowed CORS origins: {CORS_ORIGINS}")

    config = uvicorn.Config(api, host="0.0.0.0", port=API_PORT, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()
else:
    print(
        f"RUN_MODE is '{RUN_MODE}', so the API server was not started. "
        "Set the VISIONTRACK_MODE environment variable to 'api' and re-run "
        "this cell (this is what the backend Docker image does automatically) "
        "to start the FastAPI + MongoDB service that the React dashboard talks to."
    )


## 17. Performance notes

On a CPU-only environment, SSD MobileNet V2 is expected to be the main
bottleneck.

Typical baseline for this project is around **2–3 FPS**.

Optimization paths:

- process every Nth frame
- run detection less frequently and tracking more frequently
- TensorFlow Lite
- quantization
- lighter detector
- GPU / hardware acceleration

The important engineering point is that profiling showed inference, rather
than simple frame resizing, dominates latency. `/api/process-frame` also runs
detection in a thread pool (`run_in_threadpool`) so a slow frame doesn't
block the event loop from answering `/api/health` or serving other requests,
and MongoDB writes are throttled to one snapshot every
`SNAPSHOT_INTERVAL_SECONDS` so persistence never becomes the bottleneck.

## 18. Interview explanation

A concise explanation:

> I used a pretrained SSD MobileNet V2 model trained on COCO and integrated
> it into a real-time computer-vision pipeline. The notebook handles
> preprocessing, inference, confidence filtering, bounding-box conversion,
> ByteTrack multi-object tracking, object counting, line-crossing analytics,
> visualization, session-level analytics, and the FastAPI service itself. The
> rest of the project adds the React frontend, MongoDB persistence, Docker
> infrastructure, and system-design documentation around this notebook.

The model is pretrained; the project's contribution is the end-to-end system
integration — detection → tracking → analytics → a persisted, queryable
session history — and the surrounding system engineering (API design,
database schema, containerization, CORS/config handling).